# 机械工程小助手 — LoRA 微调（Kaggle 免费 GPU）

## 使用步骤
1. 打开 https://www.kaggle.com 注册/登录
2. 右上角 **Create → Notebook**
3. 右侧 **Settings** 面板里设置两项：
   - **Accelerator** → `GPU T4 x2`
   - **Internet** → `On`（必须开，否则下载不了模型）
4. 把本文件上传到 Kaggle（File → Import Notebook），或直接复制每个 cell
5. 先填第 3 个 cell 的数据，再 **Run All**

## 这个 notebook 干了什么
| 步骤 | 说明 |
|---|---|
| 加载模型 | Qwen2.5-1.5B-Instruct，fp16 放显存 |
| 构造数据 | 按 chat template 拼成 token，**只在 assistant 回答部分算 loss** |
| 挂 LoRA | 只训练 <1% 的参数（约 900 万），冻结原模型 |
| 训练 | 8~10 个 epoch，几十秒到几分钟 |
| 对比 | 同一个问题，关掉 adapter 和打开 adapter 各答一次 |
| 保存 | 导出 adapter（约 20MB），可下载回本地 |


In [ ]:
# ===== 1. 环境检查 + 装依赖 =====
!nvidia-smi

# Kaggle 自带 torch / transformers，只需补 peft
!pip install -q peft datasets accelerate

import torch, transformers
print('torch', torch.__version__, '| transformers', transformers.__version__)
print('CUDA 可用:', torch.cuda.is_available())


In [ ]:
# ===== 2. 训练数据（你要写 20~30 条）=====
#
# 格式：一条 = 一问一答。用你机械电子专业的真实知识写，别编。
# 建议覆盖：机械设计（键/带/链/齿轮/轴承）、机械原理（机构/自由度）、
#           公差配合、材料与热处理、液压/气动、PLC/单片机
#
# 下面是 3 条示例，照着格式往下加，把 TODO 区域补满 20~30 条。

DATA = [
    {'messages': [
        {'role': 'user', 'content': '键连接的作用是什么？'},
        {'role': 'assistant', 'content': '键连接用于实现轴与轮毂之间的周向固定，并传递扭矩。常见类型有平键、半圆键和楔键，其中平键装拆方便、对中性好，应用最广。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': 'V带传动为什么比平带传动承载能力高？'},
        {'role': 'assistant', 'content': 'V带靠两侧面与轮槽之间的楔形摩擦传力，当量摩擦系数约为平带的 3 倍。因此在同样张紧力下，V带不易打滑，承载能力更高。'},
    ]},
    {'messages': [
        {'role': 'user', 'content': '轴承代号 6208 表示什么意思？'},
        {'role': 'assistant', 'content': '6 表示深沟球轴承，2 表示尺寸系列为轻系列，08 表示内径为 8×5=40 mm。这是最常用的一类滚动轴承代号。'},
    ]},
    # TODO: 从这里往下继续加 20~30 条
    # {'messages': [
    #     {'role': 'user', 'content': '你的问题'},
    #     {'role': 'assistant', 'content': '你的回答'},
    # ]},
]

print(f'训练样本数：{len(DATA)} 条')
assert len(DATA) >= 20, '至少写 20 条再训练，否则模型学不到东西'


In [ ]:
# ===== 3. 加载模型和分词器 =====
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.config.use_cache = False
print('参数量：', round(model.num_parameters() / 1e9, 2), 'B')
print('设备：', next(model.parameters()).device)


In [ ]:
# ===== 4. 构造训练集 =====
# 关键点：labels 里把「提问部分」设成 -100，表示不计算损失
# 这样模型只学「怎么回答」，不学「怎么提问」
from torch.utils.data import Dataset

MAX_LEN = 512


def encode(messages):
    prompt_ids = tok.apply_chat_template(
        messages[:-1], tokenize=True, add_generation_prompt=True)
    full_ids = tok.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=False)
    full_ids = full_ids[:MAX_LEN]
    n_prompt = min(len(prompt_ids), len(full_ids))
    labels = [-100] * n_prompt + full_ids[n_prompt:]
    return {'input_ids': full_ids, 'labels': labels}


class QADataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return encode(self.rows[i]['messages'])


def collate(batch):
    max_len = max(len(b['input_ids']) for b in batch)
    pad_id = tok.pad_token_id or tok.eos_token_id
    input_ids, labels, attn = [], [], []
    for b in batch:
        pad = max_len - len(b['input_ids'])
        input_ids.append(b['input_ids'] + [pad_id] * pad)
        labels.append(b['labels'] + [-100] * pad)
        attn.append([1] * len(b['input_ids']) + [0] * pad)
    return {
        'input_ids': torch.tensor(input_ids),
        'labels': torch.tensor(labels),
        'attention_mask': torch.tensor(attn),
    }


ds = QADataset(DATA)
sample = ds[0]
n_mask = sum(1 for x in sample['labels'] if x == -100)
print(f'第 1 条总长 {len(sample["input_ids"])}，其中被屏蔽 {n_mask} 个 token')
print('校验：屏蔽数应该 > 0，说明 prompt 部分没有参与 loss')


In [ ]:
# ===== 5. 挂 LoRA =====
# r 越大能学的内容越多，但也越容易过拟合；小数据集用 8~16 足够
from peft import LoraConfig, get_peft_model, TaskType

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


In [ ]:
# ===== 6. 训练 =====
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir='/kaggle/working/ckpt',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=2,
    save_strategy='no',
    report_to='none',
    remove_unused_columns=False,
    optim='adamw_torch',
    seed=42,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds,
    data_collator=collate,
)

trainer.train()


In [ ]:
# ===== 7. loss 曲线 =====
import matplotlib.pyplot as plt

hist = [h for h in trainer.state.log_history if 'loss' in h]
steps = [h['step'] for h in hist]
losses = [h['loss'] for h in hist]

plt.figure(figsize=(7, 3))
plt.plot(steps, losses, marker='o', markersize=3)
plt.xlabel('step')
plt.ylabel('loss')
plt.title('LoRA 训练损失')
plt.grid(alpha=0.3)
plt.show()

print(f'起始 loss: {losses[0]:.4f} → 结束 loss: {losses[-1]:.4f}')
print('正常情况：整体下降，末尾有波动（数据少导致的正常现象）')


In [ ]:
# ===== 8. 微调前后对比 =====
# 用同一个问题，分别关掉 / 打开 adapter 生成
from contextlib import nullcontext


def ask(question, use_adapter=True, max_new_tokens=200):
    msgs = [{'role': 'user', 'content': question}]
    ids = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors='pt')
    ids = ids.to(model.device)
    ctx = nullcontext() if use_adapter else model.disable_adapter()
    with ctx, torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=max_new_tokens,
            do_sample=False, temperature=None, top_p=None,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


test_qs = [
    '键连接的作用是什么？',
    'V带传动为什么比平带传动承载能力高？',
    '轴承代号 6208 表示什么意思？',
]

for q in test_qs:
    print('=' * 60)
    print('问：', q)
    print('\n[微调前] ', ask(q, use_adapter=False)[:300])
    print('\n[微调后] ', ask(q, use_adapter=True)[:300])
    print()


In [ ]:
# ===== 9. 保存 adapter =====
# 只有 adapter 权重（约 20MB），下载方便
SAVE_DIR = '/kaggle/working/lora_adapter'
model.save_pretrained(SAVE_DIR)
tok.save_pretrained(SAVE_DIR)

import os
total = 0
for f in os.listdir(SAVE_DIR):
    size = os.path.getsize(os.path.join(SAVE_DIR, f)) / 1e6
    total += size
    print(f'  {f:30s} {size:8.1f} MB')
print(f'合计 {total:.1f} MB')
print('\n在 Kaggle 右侧 Output 面板可以下载整个 lora_adapter 目录')


## 下一步（可选）

### 把 adapter 合并进原模型，导出成能用 Ollama 跑的形式
```python
merged = model.merge_and_unload()
merged.save_pretrained('/kaggle/working/merged', safe_serialization=True)
tok.save_pretrained('/kaggle/working/merged')
```
合并后约 3GB。要变成 Ollama 能吃的 GGUF，还需要 llama.cpp 的 `convert_hf_to_gguf.py` 转换（这一步放到第四阶段周 15 一起做）。

### 面试怎么讲这个项目
1. **为什么用 LoRA**：全参微调 1.5B 需要约 12GB 显存存梯度和优化器状态；LoRA 只训练低秩旁路矩阵，可训练参数降到 1% 以下，T4 单卡就能跑。
2. **为什么 labels 要屏蔽 prompt**：SFT 是让模型学「给定问题怎么答」，不是学「怎么问」。不屏蔽的话模型会把提问部分也算进损失，浪费容量。
3. **r 和 alpha 怎么选**：r 是低秩矩阵的秩，决定旁路容量；alpha/r 是缩放系数，常用 2。数据少就用小 r，防过拟合。
4. **数据量为什么这么少**：这只是一个最小可跑通的 demo。真实项目通常需要几千到几万条，并且要做数据清洗、去重、质量过滤。
